In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import pandas as pd
import hashlib
import shutil


# ============================================================
# Configuration
# ============================================================

config = dotenv_values(".env")

downloads = Path(config["DOWNLOADS_PATH"])
raw_files = Path.cwd() / "raw_files" / "bitget"

raw_file_prefix = "bitget"
file_pattern = "Exportar transacciones *"


# ============================================================
# File selection
# ============================================================

def get_files(folder, pattern):
    return list(folder.glob(pattern))


# ============================================================
# Date range
# ============================================================

def get_date_range(file):
    df = pd.read_csv(file)

    dates = pd.to_datetime(
        df["Date"],
        errors="coerce"
    ).dropna()

    if dates.empty:
        raise ValueError(
            f"No valid dates found in {file.name}"
        )

    return dates.min(), dates.max()


# ============================================================
# Filename
# ============================================================

def create_new_filename(prefix, oldest_date, newest_date):
    return (
        f"{prefix}_"
        f"{oldest_date:%Y-%m-%d}_"
        f"{newest_date:%Y-%m-%d}.csv"
    )


# ============================================================
# File hash
# ============================================================

def get_file_hash(file):
    hash_sha256 = hashlib.sha256()

    with open(file, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_sha256.update(chunk)

    return hash_sha256.hexdigest()


# ============================================================
# Check whether exact file already exists
# ============================================================

def file_is_already_archived(file, raw_files):
    source_hash = get_file_hash(file)

    for archived_file in raw_files.glob("*.csv"):
        if get_file_hash(archived_file) == source_hash:
            return True

    return False


# ============================================================
# Create a unique destination filename
# ============================================================

def get_destination(raw_files, filename):
    destination = raw_files / filename

    if not destination.exists():
        return destination

    stem = destination.stem
    suffix = destination.suffix
    counter = 2

    while True:
        destination = raw_files / f"{stem}_{counter}{suffix}"

        if not destination.exists():
            return destination

        counter += 1


# ============================================================
# Archive new files
# ============================================================

raw_files.mkdir(
    parents=True,
    exist_ok=True
)

files = get_files(
    downloads,
    file_pattern
)

for file in files:

    # Skip exact duplicates
    if file_is_already_archived(file, raw_files):
        print(f"Already archived: {file.name}")
        continue

    # Get date range
    oldest_date, newest_date = get_date_range(file)

    # Create archive filename
    new_name = create_new_filename(
        raw_file_prefix,
        oldest_date,
        newest_date
    )

    # Avoid overwriting files with the same name
    destination = get_destination(
        raw_files,
        new_name
    )

    # Copy, never move
    shutil.copy2(
        file,
        destination
    )

    print(
        f"Archived: {file.name} -> {destination.name}"
    )